In [0]:
 # 03c — Gold: Conectividad por barrio + directividad de líneas

import pandas as pd
from datetime import datetime, timezone

In [0]:
CATALOG = "gobierno_abierto"
SCHEMA_SILVER = "b_silver"
SCHEMA_GOLD = "c_gold"

In [0]:
## 1. `gold.conectividad_barrio`

route_stops = spark.table(f"{CATALOG}.{SCHEMA_SILVER}.route_stops").toPandas()
stops_by_barrio = spark.table(f"{CATALOG}.{SCHEMA_GOLD}.stops_by_barrio").toPandas()
barrio_coverage = spark.table(f"{CATALOG}.{SCHEMA_GOLD}.barrio_coverage").toPandas()

lineas_por_stop = route_stops[["route_id", "stop_id"]].drop_duplicates()

lineas_por_barrio = (
    lineas_por_stop.merge(stops_by_barrio[["stop_id", "barrio_id"]], on="stop_id", how="inner")
    .dropna(subset=["barrio_id"])
    .groupby("barrio_id")["route_id"]
    .nunique()
    .rename("cantidad_lineas_distintas")
    .reset_index()
)

print(f"Barrios con al menos una línea identificada: {len(lineas_por_barrio)}")

conectividad_barrio = barrio_coverage[
    ["barrio_id", "barrio_nombre", "area_km2", "cantidad_paradas", "paradas_por_km2", "ranking_cobertura"]
].merge(lineas_por_barrio, on="barrio_id", how="left")

conectividad_barrio["cantidad_lineas_distintas"] = conectividad_barrio["cantidad_lineas_distintas"].fillna(0).astype(int)

# líneas por km² — mide densidad de conectividad, no solo de paradas
conectividad_barrio["lineas_por_km2"] = conectividad_barrio.apply(
    lambda r: r["cantidad_lineas_distintas"] / r["area_km2"] if pd.notna(r["area_km2"]) and r["area_km2"] > 0 else None,
    axis=1,
)

conectividad_barrio["paradas_por_linea"] = conectividad_barrio.apply(
    lambda r: r["cantidad_paradas"] / r["cantidad_lineas_distintas"] if r["cantidad_lineas_distintas"] > 0 else None,
    axis=1,
)

conectividad_barrio["ranking_conectividad"] = (
    conectividad_barrio["lineas_por_km2"].rank(method="min", ascending=False).astype("Int64")
)

n = conectividad_barrio["barrio_id"].nunique()
conectividad_barrio["percentil_cobertura"] = 1 - (conectividad_barrio["ranking_cobertura"] - 1) / n
conectividad_barrio["percentil_conectividad"] = 1 - (conectividad_barrio["ranking_conectividad"] - 1) / n
conectividad_barrio["desajuste_cobertura_conectividad"] = (
    conectividad_barrio["percentil_cobertura"] - conectividad_barrio["percentil_conectividad"]
)

conectividad_barrio = conectividad_barrio.sort_values("desajuste_cobertura_conectividad", ascending=False)
conectividad_barrio["_created_at"] = datetime.now(timezone.utc)

conectividad_barrio.head(10)

In [0]:
spark.createDataFrame(conectividad_barrio).write.mode("overwrite").option(
    "overwriteSchema", "true"
).saveAsTable(f"{CATALOG}.{SCHEMA_GOLD}.conectividad_barrio")

count = spark.table(f"{CATALOG}.{SCHEMA_GOLD}.conectividad_barrio").count()
print(f"conectividad_barrio: {count} filas escritas.")

In [0]:
## 2. `gold.indice_directividad_lineas`

route_directness = spark.table(f"{CATALOG}.{SCHEMA_SILVER}.route_directness").toPandas()

route_directness["ranking_directividad"] = (
    route_directness["ratio_directividad"].rank(method="min", ascending=True).astype("Int64")
)
route_directness = route_directness.sort_values("ranking_directividad")
route_directness["_created_at"] = datetime.now(timezone.utc)

circulares = route_directness["es_circular"].sum()
print(f"Líneas/shapes circulares (sin ranking de directividad): {circulares}")

route_directness.head(10)

In [0]:
spark.createDataFrame(route_directness).write.mode("overwrite").option(
    "overwriteSchema", "true"
).saveAsTable(f"{CATALOG}.{SCHEMA_GOLD}.indice_directividad_lineas")

count = spark.table(f"{CATALOG}.{SCHEMA_GOLD}.indice_directividad_lineas").count()
print(f"indice_directividad_lineas: {count} filas escritas.")

In [0]:
## Verificación final

print("Barrios con MÁS paradas relativas que líneas distintas (posible redundancia):")
display(
    spark.table(f"{CATALOG}.{SCHEMA_GOLD}.conectividad_barrio")
    .orderBy("desajuste_cobertura_conectividad", ascending=False)
    .limit(10)
)

print("\nLíneas más indirectas/sinuosas (ratio_directividad más alto):")
display(
    spark.table(f"{CATALOG}.{SCHEMA_GOLD}.indice_directividad_lineas")
    .filter("es_circular = false")
    .orderBy("ratio_directividad", ascending=False)
    .limit(10)
)
